# Student Performance Prediction System: Exploratory Data Analysis (EDA)
**Course**: VITyarthi – Fundamentals of AI and ML  
**Author**: Multi-Agent Engineering Team (Agent 2 - Data Scientist)  
**Dataset**: Student Academic Performance Benchmark (`data/raw/student_performance_data.csv`)

## Notebook Objectives
1. Profile dataset structure, dimensions, and attribute types.
2. Quantify and visualize missing values.
3. Analyze statistical distributions of numerical features and examine target variable variance.
4. Evaluate multi-variable correlations with `Final_Exam_Score`.
5. Synthesize preprocessing and feature engineering recommendations for the ML pipeline.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("Environment initialized successfully.")


### 1. Data Ingestion & Schema Inspection
We load the raw student dataset and examine column data types, non-null counts, and memory consumption.


In [ ]:
data_path = '../data/raw/student_performance_data.csv'
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.info()


### 2. First Look at the Data
Displaying the first 5 records from the raw dataset.


In [ ]:
df.head()


### 3. Missing Value Analysis
Detecting the exact count and percentage of missing values per feature.


In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage (%)': missing_pct})
missing_df[missing_df['Missing Count'] > 0]


**Analysis**:
- `Attendance_Rate` has 20 missing values (2.0%).
- `Study_Hours_Per_Week` has 18 missing values (1.8%).
- `Parental_Education_Level` has 15 missing values (1.5%).
- **Strategy**: Median imputation will be employed for continuous variables (robust against outliers), while modal (most frequent) imputation will be used for categorical attributes. Crucially, imputers must be fitted **only** on the training split to prevent data leakage.


### 4. Descriptive Statistics
Computing central tendency, dispersion, and range metrics for all continuous variables.


In [ ]:
df.describe().round(2)


### 5. Distribution of Target Variable (`Final_Exam_Score`)
Visualizing the distribution and normality of student final exam scores.


In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(df['Final_Exam_Score'], kde=True, color='#1f77b4', bins=30)
plt.axvline(df['Final_Exam_Score'].mean(), color='red', linestyle='--', label=f"Mean: {df['Final_Exam_Score'].mean():.2f}")
plt.axvline(df['Final_Exam_Score'].median(), color='green', linestyle='-', label=f"Median: {df['Final_Exam_Score'].median():.2f}")
plt.title("Distribution of Final Exam Scores", fontsize=13, fontweight='bold')
plt.xlabel("Final Exam Score (0-100)")
plt.ylabel("Student Count")
plt.legend()
plt.tight_layout()
plt.show()


**Analysis**:
The target variable exhibits a well-behaved, approximately normal distribution centered around a mean of ~66.9 with a standard deviation of ~8.9. No ceiling or floor truncation effects are present, making continuous linear and non-linear regression algorithms suitable.


### 6. Correlation Analysis
Analyzing pairwise Pearson correlation coefficients across numerical attributes.


In [ ]:
num_cols = ['Attendance_Rate', 'Study_Hours_Per_Week', 'Previous_Score',
            'Assignment_Completion_Rate', 'Internal_Assessment_Score', 'Class_Participation', 'Final_Exam_Score']

corr_matrix = df[num_cols].corr()
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, mask=mask, linewidths=0.5)
plt.title("Correlation Matrix of Academic Features", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


**Key Correlation Insights**:
1. `Internal_Assessment_Score` and `Previous_Score` show strong positive correlations with `Final_Exam_Score`.
2. `Attendance_Rate` and `Study_Hours_Per_Week` show substantial positive associations with summative performance.
3. Feature collinearity between previous marks and internal assessments indicates potential value for interaction terms (e.g., assessment momentum).


### 7. Categorical Feature Impact
Examining student performance across categorical factors: Parental Education, Internet Access, and Extra-Curriculars.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

edu_order = ['High School', 'Associate', 'Bachelor', 'Master', 'Doctorate']
sns.boxplot(data=df, x='Parental_Education_Level', y='Final_Exam_Score', order=edu_order, ax=axes[0], palette='Blues')
axes[0].set_title("Final Score by Parental Education", fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

sns.boxplot(data=df, x='Internet_Access', y='Final_Exam_Score', ax=axes[1], palette='Set2')
axes[1].set_title("Final Score by Internet Access", fontweight='bold')

sns.boxplot(data=df, x='Extra_Curricular', y='Final_Exam_Score', ax=axes[2], palette='Pastel1')
axes[2].set_title("Final Score by Extra-Curricular Activity", fontweight='bold')

plt.tight_layout()
plt.show()


### 8. Conclusions & Pipeline Directives
Based on the exploratory data analysis:
1. **Preprocessing Pipeline**: Must combine `SimpleImputer` (median for numerical, most-frequent for categorical), `StandardScaler` for continuous predictors, and `OneHotEncoder` for nominal attributes.
2. **Feature Engineering**:
   - `Academic_Engagement_Index`: Weighted synthesis of Attendance and Assignment Submission.
   - `Assessment_Momentum`: Trajectory delta between internal assessments and previous score.
   - `Study_Efficiency_Ratio`: Score output per hour invested.
3. **Modeling Candidate Algorithms**:
   - Linear Regression (Baseline)
   - Decision Tree Regressor (Non-linear interactions)
   - Random Forest Regressor (Ensemble bagging, robust against variance)
   - Gradient Boosting Regressor (Sequential error correction)
